In [15]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Configure S3 client to use unsigned requests
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = 'broad-references'
prefix = 'hg38'

paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get('Contents', []):
        print(obj['Key'])



hg38/v0/1000G.phase3.integrated.sites_only.no_MATCHED_REV.hg38.vcf
hg38/v0/1000G.phase3.integrated.sites_only.no_MATCHED_REV.hg38.vcf.idx
hg38/v0/1000G_omni2.5.hg38.vcf.gz
hg38/v0/1000G_omni2.5.hg38.vcf.gz.tbi
hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz
hg38/v0/1000G_phase1.snps.high_confidence.hg38.vcf.gz.tbi
hg38/v0/1000G_phase3_v4_20130502.sites.hg38.vcf
hg38/v0/1000G_phase3_v4_20130502.sites.hg38.vcf.idx
hg38/v0/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf.gz
hg38/v0/Axiom_Exome_Plus.genotypes.all_populations.poly.hg38.vcf.gz.tbi
hg38/v0/CrossSpeciesContamination/ContaminantNormalizationFactors.txt
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.dict
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa.fai
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.fa.img
hg38/v0/CrossSpeciesContamination/CrossSpeciesContaminant/meats.min2k.db
hg38/v0/CrossSpec

In [22]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

bucket = "1000genomes"

prefix = "1000G_2504_high_coverage/working/"

resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, Delimiter="/")

# Show subdirectories
print("=== SUBFOLDERS ===")
for p in resp.get("CommonPrefixes", []):
    print(p["Prefix"])

# Show files
print("\n=== FILES ===")
for obj in resp.get("Contents", []):
    print(obj["Key"])

=== SUBFOLDERS ===
1000G_2504_high_coverage/working/20201028_3202_phased/
1000G_2504_high_coverage/working/20201028_3202_raw_GT_with_annot/

=== FILES ===


In [5]:
import s3fs
import json

# Create an S3 filesystem object with unsigned access
fs = s3fs.S3FileSystem(anon=True)

s3_file = "s3://1000genomes/1000G_2504_high_coverage/working/20201028_3202_phased/phased-manifest_July2021.tsv"

with fs.open(s3_file, 'r') as f:
    metadata = json.load(f)

# Inspect top-level keys
print(metadata.keys())


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [11]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Create unsigned S3 client
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
prefix = "1000G_2504_high_coverage/working"

# Get list of VCFs and TBIs
print(f"Listing files under s3://{bucket}/{prefix} ...\n")
paginator = s3.get_paginator('list_objects_v2')
vcf_info = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        size = obj["Size"]
        if key.endswith(".vcf.gz") or key.endswith(".vcf.gz.tbi"):
            vcf_info.append((key, size))

# Print summary
for key, size in sorted(vcf_info):
    flag = "✅ OK" if size > 0 else "❌ EMPTY"
    print(f"{key:<100} {size/1e6:8.2f} MB   {flag}")

# Optional: Check if every VCF has a corresponding .tbi file
print("\n=== Index pairing check ===")
vcf_bases = {k.replace(".tbi", "") for k, _ in vcf_info}
for base in vcf_bases:
    has_vcf = any(k == base for k, _ in vcf_info)
    has_tbi = any(k == base + ".tbi" for k, _ in vcf_info)
    if has_vcf and not has_tbi:
        print(f"⚠️ Missing index for: {base}")


Listing files under s3://1000genomes/1000G_2504_high_coverage/working ...

1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr10.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr10.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr11.filtered.shapeit2-duohmm-phased.vcf.gz     0.00 MB   ❌ EMPTY
1000G_2504_high_coverage/working/20201028_3202_phased/CCDG_14151_B01_GRM_WGS_2020-08-05_chr11.filtered.shapeit2-duohmm-phased.vcf.gz.tbi     0.0

In [4]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

# Unsigned S3 client (for public buckets like 1000 Genomes)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

bucket = "1000genomes"
prefix = "release/20130502/ALL.chr21"

# Use paginator to list all files
paginator = s3.get_paginator('list_objects_v2')
vcf_info = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get('Contents', []):
        key = obj['Key']
        if key.endswith(".vcf.gz.tbi"):
            # Check if the index exists by looking for a .tbi file in the same prefix
            index_key = key + ".tbi"
            try:
                s3.head_object(Bucket=bucket, Key=index_key)
                has_index = True
            except s3.exceptions.ClientError:
                has_index = False

            vcf_info.append({
                "vcf": key,
                "size_MB": obj['Size'] / 1024**2,
                "has_index": has_index
            })

# Print results
for info in vcf_info:
    print(f"{info['vcf']}  {info['size_MB']:.2f} MB  Index: {info['has_index']}")


release/20130502/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz.tbi  0.03 MB  Index: False


In [31]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import gzip
import io

# Setup S3 client
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED)
)

bucket = "1000genomes"

# =========================
# CONFIG: change these keys
# =========================
keys_to_check = [
    # Phase 3 (likely hg19)
    "release/20130502/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz",

    # High coverage (likely hg38)
    "1000G_2504_high_coverage/working/20201028_3202_raw_GT_with_annot/20201028_CCDG_14151_B01_GRM_WGS_2020-08-05_chr21.recalibrated_variants.vcf.gz"
]


# =========================
# FUNCTION
# =========================
def inspect_vcf(key):
    print("\n" + "="*80)
    print("FILE:", key)

    # 1. HEAD check
    try:
        head = s3.head_object(Bucket=bucket, Key=key)
        size = head["ContentLength"]
        print("Exists: YES")
        print("Size (MB):", round(size / 1024**2, 2))
    except Exception as e:
        print("Exists: NO")
        print(e)
        return

    # 2. Quick gzip check
    try:
        obj = s3.get_object(Bucket=bucket, Key=key, Range="bytes=0-3")
        first4 = obj["Body"].read()
        print("First bytes:", first4)
        print("Gzip:", first4[:2] == b"\x1f\x8b")
    except Exception as e:
        print("Gzip check failed:", e)
        return

    # 3. Read header safely
    try:
        obj = s3.get_object(Bucket=bucket, Key=key, Range="bytes=0-5000000")
        raw = obj["Body"].read()

        with gzip.GzipFile(fileobj=io.BytesIO(raw)) as gz:
            print("\n--- HEADER ---")
            chrom_style = None

            for i, line in enumerate(gz):
                line = line.decode("utf-8", errors="replace").strip()
                print(line)

                # detect chromosome style
                if not line.startswith("#") and chrom_style is None:
                    chrom = line.split("\t")[0]
                    chrom_style = chrom

                if line.startswith("#CHROM"):
                    break

                if i > 100:
                    break

            print("\n--- BUILD HEURISTIC ---")
            if chrom_style:
                print("First CHROM:", chrom_style)
                if chrom_style.startswith("chr"):
                    print("Likely: GRCh38 (hg38)")
                else:
                    print("Likely: GRCh37 (hg19)")
            else:
                print("Could not detect chromosome style")

    except Exception as e:
        print("Header read failed:", e)


# =========================
# RUN
# =========================
for key in keys_to_check:
    inspect_vcf(key)


# =========================
# OPTIONAL: scan directory
# =========================
print("\n" + "="*80)
print("SCANNING DIRECTORY FOR VCF FILES")

prefix = "1000G_2504_high_coverage/working/"

paginator = s3.get_paginator("list_objects_v2")

count = 0
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        size = obj["Size"]

        if key.endswith(".vcf.gz") and size > 0:
            print(round(size/1024**2,2), "MB", key)
            count += 1

            if count > 50:
                break
    if count > 50:
        break

print("\nTotal shown:", count)


FILE: release/20130502/ALL.chr21.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz
Exists: YES
Size (MB): 208.49
First bytes: b'\x1f\x8b\x08\x04'
Gzip: True

--- HEADER ---
##fileformat=VCFv4.1
##FILTER=<ID=PASS,Description="All filters passed">
##fileDate=20150218
##reference=ftp://ftp.1000genomes.ebi.ac.uk//vol1/ftp/technical/reference/phase2_reference_assembly_sequence/hs37d5.fa.gz
##source=1000GenomesPhase3Pipeline
##contig=<ID=1,assembly=b37,length=249250621>
##contig=<ID=2,assembly=b37,length=243199373>
##contig=<ID=3,assembly=b37,length=198022430>
##contig=<ID=4,assembly=b37,length=191154276>
##contig=<ID=5,assembly=b37,length=180915260>
##contig=<ID=6,assembly=b37,length=171115067>
##contig=<ID=7,assembly=b37,length=159138663>
##contig=<ID=8,assembly=b37,length=146364022>
##contig=<ID=9,assembly=b37,length=141213431>
##contig=<ID=10,assembly=b37,length=135534747>
##contig=<ID=11,assembly=b37,length=135006516>
##contig=<ID=12,assembly=b37,length=133851895>
##cont